<a href="https://colab.research.google.com/github/oliveirasWell/ufms-redes-neurais-2025/blob/feat%2Frag/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG - Retrieval-Augmented Generation

Sistema de RAG (Retrieval-Augmented Generation) para responder perguntas sobre o [regulamento da pós graduação da FACOM UFMS](https://www.facom.ufms.br/wp-content/uploads/2023/07/20232-EM-DIANTE-ATUAL-RESOLUCAO__COPP__n_704__de_07_07_2023.pdf)

A implementação tem como objetivo:
- Extração e processamento de texto de documentos PDF
- Criação de embeddings e armazenamento em banco vetorial (ChromaDB)
- Sistema de recuperação de contexto relevante baseado em similaridade semântica
- Geração de respostas usando modelo de linguagem (Ollama) com contexto recuperado
- Avaliação de precisão factual das respostas geradas

## Metodologia

A metodologia consiste em:

1. Download e extração de texto do PDF do regulamento
2. Limpeza e normalização do texto extraído
3. Divisão do texto em chunks com diferentes estratégias (básica e avançada)
4. Geração de embeddings usando modelos multilíngues (SentenceTransformers)
5. Armazenamento dos embeddings e documentos no ChromaDB
6. Recuperação de k contextos mais relevantes para cada pergunta
7. Geração de respostas usando Ollama com contexto recuperado
8. Avaliação de fact score para diferentes valores de k e estratégias de chunking
9. Comparação e análise dos resultados

### Detalhes de implementação

- Foi utilizado o modelo [llama3.2:3b](https://ollama.com/library/llama3.2) via Ollama para geração de respostas
- Modelos de embedding multilíngues: `paraphrase-multilingual-MiniLM-L12-v2` (padrão), `paraphrase-multilingual-mpnet-base-v2` e `distiluse-base-multilingual-cased-v2`
- Banco vetorial ChromaDB com espaço de similaridade cosseno
- Duas estratégias de chunking: básica (divisão simples com overlap) e avançada (respeita estrutura do documento, divisão por sentenças)
- Parâmetros configuráveis: `CHUNK_SIZE` (400, 800), `BATCH_SIZE` (64), `k` (número de contextos recuperados: 1, 5, 10, 20, 30, 40, 50)
- Extração de PDF usando `pdfplumber`
- Limpeza de texto remove números de página, datas, normaliza unicode e preserva estrutura (Art., CAPÍTULO, etc)
- Avaliação de fact score calcula precisão factual das respostas em relação ao contexto recuperado
- O código pode ser executado em CPU, GPU ou MPS (Macbooks) - detecta automaticamente o dispositivo disponível

### Execução 

- Este código é executado em um servidor jupyter
- Requer instalação do Ollama e modelo llama3.2:3b
- Configure `CHUNK_SIZE`, `chunking_type` e `k` para diferentes experimentos
- Os resultados de fact score são plotados em gráficos de barras para análise comparativa



In [ ]:
import requests

In [ ]:
#Only run if system == "Linux":
# Step 1: system + Python deps
!pip -q install "mcp[cli]" fastapi uvicorn pytz requests nest_asyncio ollama pdfplumber chromadb sentence_transformers

In [ ]:
# # Step 2: install Ollama (Linux)
!curl -fsSL https://ollama.com/install.sh | sh
# # Step 3: start Ollama server in background
!nohup ollama serve >/dev/null 2>&1 &
# # Step 4: pull a tiny model so Colab doesn't choke
!ollama pull llama3.2:3b

In [ ]:
sizes = [1, 5, 10, 20, 30, 40, 50]

In [ ]:
COLLECTION_NAME = "reg_pos_facom"

In [ ]:
OLLAMA_URL = '127.0.0.1:11434'
MODEL = "llama3.2:3b"

from ollama import Client
client_ollama = Client(
  host=OLLAMA_URL
)

In [ ]:
def get_rag_response(context: str):
  response = client_ollama.chat(model=MODEL, messages=[
    {
      'role': 'system',
      'content': "Você é um assistente de IA que responde perguntas sobre o regulamento da pós graduação da Faculdade de Computação da Universidade Federal de Mato Grosso do Sul - FACOM UFMS."
    },
    {
      'role': 'user',
      'content': context,
    },
  ])
  return response.message.content


In [ ]:
import os
from pathlib import Path

def download_pdf_if_not_exists(url: str, local_path: str) -> str:
    """
    Download PDF from URL if it doesn't exist locally.

    Args:
        url: URL to download the PDF from
        local_path: Local path where the PDF should be saved

    Returns:
        Local path to the PDF file
    """
    local_path_obj = Path(local_path)

    # Create directory if it doesn't exist
    local_path_obj.parent.mkdir(parents=True, exist_ok=True)

    # Check if file already exists
    if local_path_obj.exists():
        print(f"PDF already exists at {local_path}, skipping download.")
        return local_path

    # Download the PDF
    print(f"Downloading PDF from {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    # Save to local path
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"PDF saved to {local_path}")
    return local_path


In [ ]:
import unicodedata
import pdfplumber
import re

def extract_text_from_pdf(pdf_path: str) -> str:
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    full_text = "\n\n".join(pages_text)
    return full_text

def basic_cleanup(text: str) -> str:
    text = re.sub(r"N°\s*\d+", "", text)
    text = re.sub(r"Pg\.\s*\d+", "", text)
    text = re.sub(r"\d{2}/\d{2}/\d{4}", "", text)
    text = re.sub(r"BOLETIM.*?\n", "", text, flags=re.IGNORECASE)

    text = text.replace('\x00', 't')
    text = text.replace('\u0000', 't')

    nfd = unicodedata.normalize('NFD', text)
    text = ''.join(char for char in nfd if unicodedata.category(char) != 'Mn')

    text = re.sub(r"[ \t]+", " ", text)

    paragraph_start = r'(Art\.|CAPÍTULO|§|Parágrafo único|I[IVX]*\s*[–-]|MARIA|ANEXO|RESOLUÇÃO)'
    text = re.sub(r'([^.!?\n])\s*\n(?!\s*' + paragraph_start + ')', r'\1 ', text, flags=re.IGNORECASE)

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"^\n+", "", text)
    text = text.strip()

    return text

In [ ]:
CHUNK_SIZE = 800

def break_text_into_chunks(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    overlap = int(chunk_size * 0.25)
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]

        if end < text_length:
            for punct in ['. ', '.\n', '! ', '!\n', '? ', '?\n']:
                last_punct = chunk.rfind(punct)
                if last_punct > chunk_size * 0.7:
                    chunk = chunk[:last_punct + len(punct)]
                    end = start + len(chunk)
                    break

        chunks.append(chunk.strip())
        start = end - overlap

    return chunks

def advanced_chunking(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap_ratio: float = 0.2,
    respect_structure: bool = True
) -> list[str]:
    """
    Chunking avançado que combina:
    1. Respeito à estrutura (Art., CAPÍTULO)
    2. Divisão por sentenças completas
    3. Overlap inteligente
    """
    overlap = int(chunk_size * overlap_ratio)

    if respect_structure:
        structure_pattern = r'(\n\n?(?:CAPÍTULO|Art\.|§\s+\d+|Parágrafo único)[^\n]*)'
        parts = re.split(structure_pattern, text)
        text_segments = []

        for i, part in enumerate(parts):
            if part.strip():
                if i > 0 and re.match(structure_pattern, parts[i-1]):
                    text_segments.append(parts[i-1] + part)
                else:
                    text_segments.append(part)
    else:
        text_segments = [text]

    chunks = []

    for segment in text_segments:
        if len(segment) <= chunk_size:
            chunks.append(segment.strip())
            continue

        sentence_pattern = r'([.!?]+[\s\n]+)'
        parts = re.split(sentence_pattern, segment)
        sentences = []

        for i in range(0, len(parts), 2):
            if i < len(parts):
                sent = parts[i]
                if i + 1 < len(parts):
                    sent += parts[i + 1]
                if sent.strip():
                    sentences.append(sent.strip())

        current_chunk = ""

        for sentence in sentences:
            test_chunk = current_chunk + " " + sentence if current_chunk else sentence

            if len(test_chunk) <= chunk_size:
                current_chunk = test_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                    overlap_words = int(overlap / 10)
                    overlap_text = " ".join(current_chunk.split()[-overlap_words:]) if overlap_words > 0 else ""
                    current_chunk = overlap_text + " " + sentence if overlap_text else sentence
                else:
                    if len(sentence) > chunk_size:
                        words = sentence.split()
                        temp_chunk = ""
                        for word in words:
                            if len(temp_chunk + " " + word) <= chunk_size:
                                temp_chunk = temp_chunk + " " + word if temp_chunk else word
                            else:
                                if temp_chunk:
                                    chunks.append(temp_chunk.strip())
                                temp_chunk = word
                        current_chunk = temp_chunk
                    else:
                        current_chunk = sentence

        if current_chunk:
            chunks.append(current_chunk.strip())

    return [c for c in chunks if c and len(c) > 20]



In [ ]:
import torch

def get_device_info() -> str:
    if torch.cuda.is_available():
        device = "cuda"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    return device


In [ ]:
get_device_info()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

MODEL_MULTILINGUAL = "paraphrase-multilingual-mpnet-base-v2"
MODEL_MULTILINGUAL_MINI = "paraphrase-multilingual-MiniLM-L12-v2"
MODEL_MULTILINGUAL_CASED = "distiluse-base-multilingual-cased-v2"
BATCH_SIZE = 64

def generate_embeddings(text_chunks: list[str], model_name: str = MODEL_MULTILINGUAL_MINI, batch_size: int = BATCH_SIZE) -> np.ndarray:
    model = SentenceTransformer(model_name)

    embeddings = model.encode(text_chunks, show_progress_bar=True, batch_size=batch_size)
    return embeddings, model


In [ ]:
def create_ids_and_metadatas(text_chunks: list[str], source: str = "reg_pos_facom") -> tuple[list[str], list[dict]]:
    ids = []
    metadatas = []
    for i, chunk in enumerate(tqdm(text_chunks, desc="Indexando chunks")):
        ids.append(f"chunk_{i}")
        metadatas.append({"source": source, "chunk_id": i})
    return ids, metadatas

In [ ]:
import chromadb
from chromadb.config import Settings

def add_documents_to_collection(collection, ids: list[str], documents: list[str], embeddings: np.ndarray, metadatas: list[dict]):
    existing_ids = collection.get()['ids']
    if existing_ids:
        collection.delete(ids=existing_ids)

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )

client = chromadb.Client(Settings(anonymized_telemetry=False))

try:
    client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)


In [ ]:
def normalize_text(text: str) -> str:
    nfd = unicodedata.normalize('NFD', text)
    return ''.join(char for char in nfd if unicodedata.category(char) != 'Mn')

def get_k_results(model: SentenceTransformer, query: str, k: int = 5) -> list[dict]:
    normalized_query = normalize_text(query)
    results = collection.query(
        query_embeddings=model.encode(normalized_query).tolist(),
        n_results=k
    )
    return results

In [ ]:
def setup_system_prompt(context: str, question: str):
    return f"""Responda a pergunta usando APENAS as informações do contexto fornecido.

CONTEXTO:
{context}

PERGUNTA:
{question}

INSTRUÇÕES:
- Use apenas informações presentes no contexto
- Seja objetivo e responda em português formal
- Se não houver informação suficiente, responda: "Informação não encontrada no regulamento"
- Cite trechos relevantes quando possível"""


In [ ]:
questions = [
    "Quais grupos são presentes no regulamento e seus respectivos nomes?",
    "O que é o estágio?",
    "É possível covalidar créditos?",
    "Um estudante de mestrado precisa de quantos créditos no total e como devem ser distribuídos entre os grupos? Pode usar artigo científico para créditos?",
    "Até quando um estudante pode comprovar proficiência em inglês no mestrado e no doutorado? E quais são as exceções para estrangeiros?",
    "Em quais situações um estudante pode ser desligado do curso?",
    "Um artigo científico publicado antes do ingresso pode ser usado para créditos?",
    "Qual a diferença entre estudante especial e estudante ouvinte?",
    "O que preciso para solicitar qualificação de mestrado e doutorado?",
    "Qual a prazo máximo de conclusão do mestrado e do doutorado?"
]

In [ ]:
question = questions[9]

In [ ]:
def get_answer(model: SentenceTransformer, question: str, k: int = 20):
    results = get_k_results(model, query=question, k=k)
    context_text = "\n\n".join(results['documents'][0])
    system_prompt = setup_system_prompt(context_text, question)
    answer = get_rag_response(system_prompt)

    return answer, context_text

In [ ]:

def setup_model_pipeline(
    model_name: str = MODEL_MULTILINGUAL_MINI,
    batch_size: int = BATCH_SIZE,
    chunk_size: int = CHUNK_SIZE,
    chunking_type: str = "basic",
):
    pdf_url = "https://www.facom.ufms.br/wp-content/uploads/2023/07/20232-EM-DIANTE-ATUAL-RESOLUCAO__COPP__n_704__de_07_07_2023.pdf"
    pdf_path = "data/20232-EM-DIANTE-ATUAL-RESOLUCAO__COPP__n_704__de_07_07_2023.pdf"
    download_pdf_if_not_exists(pdf_url, pdf_path)
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = basic_cleanup(raw_text)

    with open("regulamento_facomp.txt", "w", encoding="utf-8") as f:
        f.write(cleaned_text)

    if chunking_type == "advanced":
        text_chunks = advanced_chunking(cleaned_text, chunk_size=chunk_size)
    else:
        text_chunks = break_text_into_chunks(cleaned_text, chunk_size=chunk_size)

    print(f"{len(text_chunks)} chunks")

    embeddings, model = generate_embeddings(text_chunks, model_name=model_name, batch_size=batch_size)

    print(f"Shape: {embeddings.shape}")
    print(f"Dimension: {embeddings.shape[1]}")

    ids, metadatas = create_ids_and_metadatas(text_chunks)

    add_documents_to_collection(collection, ids, text_chunks, embeddings, metadatas)

    return model


In [ ]:
model = setup_model_pipeline(
    chunking_type="basic"
)

for k in sizes:
    answer, _ = get_answer(model, question, k=k)
    print("k: ", k, "-"*100)
    print(answer)

In [ ]:
import re

def compute_fact_score(answer: str, context: str) -> float:
    prompt = f"""Avalie a precisão factual da resposta em relação ao contexto.

RESPOSTA:
{answer}

CONTEXTO:
{context}

TAREFA:
Para cada afirmação na resposta, classifique como:
- SUPPORTED: afirmação presente no contexto
- CONTRADICTED: afirmação contradiz o contexto
- NOT_FOUND: afirmação não encontrada no contexto

Calcule: (SUPPORTED - 2×CONTRADICTED) ÷ total_afirmações

Retorne apenas o número final entre 0 e 1."""

    response = client_ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Você é um avaliador de precisão factual - fact score"},
            {"role": "user", "content": prompt}]
    )

    numbers = re.findall(r"0?\.\d+|1\.0+|[01]", response.message.content)
    return float(numbers[0]) if numbers else 0.0


In [ ]:
def evaluate_fact_scores(model: SentenceTransformer, questions: list[str], sizes: list[int]) -> dict[int, float]:
    score_map = {}

    for k in sizes:
        mean_score = 0
        for question in questions:
            answer, context = get_answer(model, question, k=k)
            score = compute_fact_score(answer, context)
            mean_score += score

        mean_score /= len(questions)
        score_map[k] = mean_score
        print(f"k: {k}, mean_score: {mean_score}")

    return score_map

In [ ]:
import matplotlib.pyplot as plt

def plot_fact_scores(score_map: dict[int, float], title: str):
    plt.figure(figsize=(10, 6))
    plt.bar(score_map.keys(), score_map.values(), color='steelblue', width=2)
    plt.xlabel('k (número de contextos)', fontsize=12)
    plt.ylabel('Fact Score', fontsize=12)
    plt.title('Fact Score por número de contextos (k) - ' + title, fontsize=14, fontweight='bold')
    plt.xticks(list(score_map.keys()))
    plt.ylim(0, 1)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
## Pipeline with basic chunking - 400

model = setup_model_pipeline(
    model_name=MODEL_MULTILINGUAL_MINI,
    batch_size=BATCH_SIZE,
    chunk_size=400,
    chunking_type="basic"
)
score_map = evaluate_fact_scores(model, questions, sizes)
plot_fact_scores(score_map, "Basic Chunking - 400")

In [ ]:
## Pipeline with basic chunking - 800

model = setup_model_pipeline(
    model_name=MODEL_MULTILINGUAL_MINI,
    batch_size=BATCH_SIZE,
    chunk_size=CHUNK_SIZE,
    chunking_type="basic"
)
score_map = evaluate_fact_scores(model, questions, sizes)
plot_fact_scores(score_map, "Basic Chunking - 800")

In [ ]:
## Pipeline with advanced chunking

model = setup_model_pipeline(
    model_name=MODEL_MULTILINGUAL_MINI,
    batch_size=BATCH_SIZE,
    chunk_size=CHUNK_SIZE,
    chunking_type="advanced"
)
score_map = evaluate_fact_scores(model, questions, sizes)
plot_fact_scores(score_map, "Advanced Chunking")